# 🐖 07 | Evaluación independiente Mask R-CNN en test

**RECURSO: GPU T4 o superior**. Carga el modelo 06, calcula mAP COCO de segmentación una sola vez en las fechas reservadas, y genera resultados por imagen más figuras de ejemplo.

**Proyecto:** Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖

Los notebooks son independientes: cada uno utiliza los productos guardados por la etapa anterior, sin repetir descarga o entrenamiento.


## Regla de integridad

**Ejecutar este notebook solamente cuando el entrenamiento 06 esté finalizado**. El test permanece reservado. Si la evaluación ya existe y coincide el hash de pesos y YAML, no vuelve a consultar test; únicamente lee la evaluación guardada. No modificar la arquitectura en función de los resultados de prueba.

Se exportan **todas** las 373 imágenes como filas de Excel con conteos y TP/FP/FN @ IoU 0,5; solo se guardan **12 imágenes ilustrativas** en PNG para no sobrecargar Drive. Las métricas de segmentación COCO y el conteo a confianza 0,25 no son equivalentes a número físico de animales distintos.


In [ ]:
import torch
if not torch.cuda.is_available():raise RuntimeError('Activa GPU T4 antes de comenzar este notebook.')
print('✅ GPU',torch.cuda.get_device_name(0))


In [ ]:
%pip -q install "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" openpyxl pillow opencv-python-headless matplotlib


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, os

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError('No encuentro la carpeta del proyecto. Comprueba que Colab monta la cuenta correcta y que el nombre no cambió: '+str(ROOT))
DATA=ROOT/'01_Datasets'/'FaroPigSeg'
EXTRACT=DATA/'extraido'/'FaroPigSeg'
SPLIT=DATA/'split_fecha_v1'
RES=ROOT/'03_Resultados'
EXP=ROOT/'05_Exportaciones_ZIP'
for s in ['01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento']:
    (RES/s).mkdir(parents=True,exist_ok=True)
EXP.mkdir(exist_ok=True,parents=True)
MANIFEST=SPLIT/'particion_fecha_manifest.json'
YAML=SPLIT/'data_fecha.yaml'
if not (MANIFEST.is_file() and YAML.is_file()):
    raise FileNotFoundError('Faltan los archivos de partición congelada; ejecutar notebook 02 en CPU antes de continuar.')
part=json.loads(MANIFEST.read_text(encoding='utf-8'))
assert len(set(part['asignaciones_fecha'].values())-{'train','val','test'})==0
print('Carpeta del proyecto:',ROOT)
print('Fechas por grupo:',{k:sum(v==k for v in part['asignaciones_fecha'].values()) for k in ('train','val','test')})


In [ ]:
import numpy as np, pandas as pd,torch,cv2,yaml,random,math
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision.transforms.functional import to_tensor

class PigMasks(Dataset):
    def __init__(self,imagelist):
        self.paths=[Path(s.strip()) for s in Path(imagelist).read_text(encoding='utf8').splitlines() if s.strip()]
        if not self.paths:raise RuntimeError('La partición está vacía: '+str(imagelist))
        missing=[str(p) for p in self.paths if not p.is_file()]
        if missing: raise FileNotFoundError('Hay imágenes ausentes: '+str(missing[:3]))
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        p=self.paths[index]
        rgb=np.asarray(Image.open(p).convert('RGB'))
        h,w=rgb.shape[:2]
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lab.is_file():raise FileNotFoundError('Etiqueta no localizada: '+str(lab))
        masks=[];boxes=[]
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip(): continue
            vals=row.split()
            if int(vals[0])!=0:raise ValueError('Clase inesperada en '+str(lab))
            points=np.asarray([float(v) for v in vals[1:]],dtype=np.float32)
            if len(points)<6 or len(points)%2:raise ValueError('Polígono inválido en '+str(lab))
            if not np.isfinite(points).all() or np.any((points<0)|(points>1)):
                raise ValueError('Coordenadas no normalizadas en '+str(lab))
            coords=points.reshape(-1,2)*np.asarray([w,h],dtype=np.float32)
            xy=np.rint(coords).astype(np.int32)
            xy[:,0]=np.clip(xy[:,0],0,w-1);xy[:,1]=np.clip(xy[:,1],0,h-1)
            mask=np.zeros((h,w),np.uint8)
            cv2.fillPoly(mask,[xy],1)
            yy,xx=np.where(mask)
            if len(xx)<3: continue
            x0,x1=int(xx.min()),int(xx.max()+1);y0,y1=int(yy.min()),int(yy.max()+1)
            if x1<=x0 or y1<=y0:continue
            masks.append(mask);boxes.append([x0,y0,x1,y1])
        if not masks:raise RuntimeError('Imagen sin máscaras válidas: '+p.name)
        tar={'boxes':torch.as_tensor(boxes,dtype=torch.float32),
             'labels':torch.ones((len(masks),),dtype=torch.int64),
             'masks':torch.as_tensor(np.stack(masks),dtype=torch.uint8),
             'image_id':torch.tensor([index],dtype=torch.int64),
             'area':torch.as_tensor([int(m.sum()) for m in masks],dtype=torch.float32),
             'iscrowd':torch.zeros((len(masks),),dtype=torch.int64)}
        return to_tensor(rgb.copy()),tar

def collate_fn(batch):return tuple(zip(*batch))

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_maskrcnn(pretrained=True):
    model=maskrcnn_resnet50_fpn(weights='DEFAULT' if pretrained else None,
                               weights_backbone=None,min_size=512,max_size=768,
                               trainable_backbone_layers=3)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,2)
    in_channels=model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor=MaskRCNNPredictor(in_channels,256,2)
    # Congelar los mismos niveles, tanto en arranque con pesos COCO como en reanudación sin descarga.
    for name,parameter in model.backbone.body.named_parameters():
        if name.startswith(('conv1.', 'bn1.', 'layer1.')):
            parameter.requires_grad_(False)
    model.roi_heads.score_thresh=0.001
    model.roi_heads.detections_per_img=100
    return model

def config_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils
from contextlib import redirect_stdout
import io

def encode_mask(arr):
    enc=maskutils.encode(np.asfortranarray(np.asarray(arr,dtype=np.uint8)))
    enc['counts']=enc['counts'].decode('ascii')
    return enc

def coco_metric_for_model(model,dataset,device,desc='Validación',include_rows=False):
    model.eval();gts=[];preds=[];image_records=[];annotations=[];aid=1
    for i in range(len(dataset)):
        image,target=dataset[i]
        height,width=image.shape[1:]
        gts.append({'id':i+1,'width':width,'height':height,'file_name':dataset.paths[i].name})
        gt_masks=target['masks'].numpy()
        for m in gt_masks:
            encoded=encode_mask(m)
            x,y,w,h=[float(v) for v in maskutils.toBbox({'size':encoded['size'],'counts':encoded['counts'].encode('ascii')})]
            annotations.append({'id':aid,'image_id':i+1,'category_id':1,'segmentation':encoded,
                                'bbox':[x,y,w,h],'area':int(m.sum()),'iscrowd':0});aid+=1
        with torch.no_grad():
            out=model([image.to(device)])[0]
        scores=out['scores'].detach().cpu().numpy()
        ms=(out['masks'].detach().cpu().numpy()[:,0]>.5) if len(scores) else np.zeros((0,height,width),dtype=bool)
        for j,score in enumerate(scores):
            if score<0.001: continue
            m=ms[j].astype(np.uint8)
            if not m.any():continue
            enc=encode_mask(m)
            preds.append({'image_id':i+1,'category_id':1,'segmentation':enc,'score':float(score)})
        if include_rows:
            filtered=np.where(scores>=.25)[0]
            predm=[ms[k].astype(bool) for k in filtered if k<len(ms)]
            p_scores=[float(scores[k]) for k in filtered if k<len(ms)]
            pairs=[]
            for gi,g in enumerate(gt_masks):
                for pj,pm in enumerate(predm):
                    inter=np.logical_and(g,pm).sum();union=np.logical_or(g,pm).sum()
                    iou=inter/max(1,union)
                    if iou>=.5:pairs.append((float(iou),gi,pj))
            used_gt=set();used_pred=set()
            for _,gi,pj in sorted(pairs,reverse=True):
                if gi not in used_gt and pj not in used_pred:
                    used_gt.add(gi);used_pred.add(pj)
            tp=len(used_gt)
            image_records.append({'imagen':dataset.paths[i].name,
                'fecha':dataset.paths[i].name[:10],
                'anotaciones_GT':len(gt_masks),'detecciones_conf_025':len(predm),
                'TP_IoU05':tp,'FP_IoU05':len(predm)-tp,'FN_IoU05':len(gt_masks)-tp,
                'confianza_media_pred':np.mean(p_scores) if p_scores else np.nan})
        if (i+1)%50==0:print(desc,i+1,'/',len(dataset))
    gt=COCO()
    gt.dataset={'images':gts,'annotations':annotations,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}
    with redirect_stdout(io.StringIO()):
        gt.createIndex()
        dt=gt.loadRes(preds) if preds else gt.loadRes([])
        ev=COCOeval(gt,dt,'segm');ev.evaluate();ev.accumulate();ev.summarize()
    return {'seg_map50_95':float(ev.stats[0]),'seg_map50':float(ev.stats[1]),
            'seg_map75':float(ev.stats[2]),'gt':len(annotations),
            'num_images':len(dataset),'pred_coarse':len(preds)},pd.DataFrame(image_records)


In [ ]:
from matplotlib import pyplot as plt
import zipfile,platform,sys,torchvision
RUN=RES/'06_Entrenamiento'/'MaskRCNN_fecha_v1'
BEST=RUN/'maskrcnn_best_model.pt'
DONE=RUN/'ENTRENAMIENTO_FINALIZADO.json'
if not BEST.is_file() or not DONE.is_file():raise FileNotFoundError('Falta Mask R-CNN entrenado; ejecutar notebook 06 con GPU.')
CHECK=json.loads(DONE.read_text(encoding='utf8'))
if CHECK.get('yaml_sha256')!=config_hash(YAML):raise RuntimeError('Cambió el YAML desde entrenamiento. Detener.')
weights_sha=hashlib.sha256(BEST.read_bytes()).hexdigest()
CACHED=RES/'05_Reportes'/'CERDOS_07_Evaluacion_MaskRCNN.json'
EXCEL=RES/'03_Metricas'/'CERDOS_07_Por_Imagen_MaskRCNN.xlsx'
OUT=RES/'02_Segmentacion'/'MaskRCNN_fecha_v1'
OUT.mkdir(parents=True,exist_ok=True)
config=yaml.safe_load(YAML.read_text(encoding='utf8'))
test_list=Path(config['test']).read_text(encoding='utf8').splitlines()
print('TEST imágenes:',len([x for x in test_list if x.strip()]))
if CACHED.exists():
    rec=json.loads(CACHED.read_text(encoding='utf8'))
    if rec['weights_sha256']!=weights_sha or rec['yaml_sha256']!=config_hash(YAML):
        raise RuntimeError('Evaluación existente de otro modelo/configuración. No sobrescribir test.')
    if not EXCEL.is_file():raise FileNotFoundError('Resultados JSON existentes pero Excel incompleto; requiere auditoría, sin recomputar test automáticamente.')
    print('✅ Reutilizando test congelado, sin volver a evaluar.');print(rec)
else:
    device=torch.device('cuda:0')
    model=build_maskrcnn(pretrained=False).to(device)
    model.load_state_dict(torch.load(BEST,map_location='cpu',weights_only=False)['model'])
    test=PigMasks(config['test'])
    metrics,rows=coco_metric_for_model(model,test,device,desc='TEST FINAL',include_rows=True)
    if len(rows)!=len(test):raise RuntimeError('Resultados incompletos por imagen')
    per_date=(rows.groupby('fecha',as_index=False)[['anotaciones_GT','detecciones_conf_025','TP_IoU05','FP_IoU05','FN_IoU05']].sum())
    per_date['precision_iou05']=per_date['TP_IoU05']/(per_date['TP_IoU05']+per_date['FP_IoU05']).replace(0,np.nan)
    per_date['recall_iou05']=per_date['TP_IoU05']/(per_date['TP_IoU05']+per_date['FN_IoU05']).replace(0,np.nan)
    report={'etapa':'07_test_una_vez','fecha_utc':datetime.now(timezone.utc).isoformat(),
        'modelo':'Mask R-CNN ResNet50-FPN','weights_sha256':weights_sha,
        'yaml_sha256':config_hash(YAML),**metrics,'conf_threshold_conteo':0.25,
        'match_iou':0.50,'notas':['El test se separó por fecha; no asegura independencia entre animales o granjas.',
        'Diferencias entre evaluadores COCO y Ultralytics exigen precaución al comparar mAP.',
        'Detecciones por imagen no son animales únicos.']}
    with pd.ExcelWriter(EXCEL,engine='openpyxl') as w:
        pd.DataFrame([report]).to_excel(w,sheet_name='Metricas_test',index=False)
        rows.to_excel(w,sheet_name='Todas_las_imagenes',index=False)
        per_date.to_excel(w,sheet_name='Errores_por_fecha',index=False)
    CACHED.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
    print('✅ Resultados globales:',metrics)
    # Doce ilustraciones, las demás imágenes sí están incluidas en evaluación cuantitativa.
    ids=np.unique(np.linspace(0,len(test)-1,min(12,len(test)),dtype=int))
    model.eval()
    for j in ids:
        image,gt=test[int(j)]
        with torch.no_grad():pred=model([image.to(device)])[0]
        scores=pred['scores'].detach().cpu().numpy()
        masks=pred['masks'].detach().cpu().numpy()[:,0]>.5 if len(scores) else np.zeros((0,*image.shape[1:]),bool)
        original=(image.permute(1,2,0).numpy()*255).clip(0,255).astype(np.uint8).copy()
        overlay=original.copy();n=0
        for k,s in enumerate(scores):
            if s<0.25:continue
            n+=1
            m=masks[k].astype(bool)
            color=np.asarray([43,165,109],dtype=np.uint8)
            overlay[m]=((overlay[m].astype(np.float32)*.55)+(color*.45)).astype(np.uint8)
        fig,ax=plt.subplots(figsize=(8,5))
        ax.imshow(overlay);ax.axis('off')
        ax.set_title(f'{test.paths[int(j)].name[:38]} | máscaras={n}',fontsize=9)
        fig.tight_layout()
        fig.savefig(OUT/f'{test.paths[int(j)].stem}_maskrcnn.png',dpi=140,bbox_inches='tight')
        plt.close(fig)
    del model; torch.cuda.empty_cache()


In [ ]:
report=json.loads(CACHED.read_text(encoding='utf8'))
print('Mask R-CNN mAP50 máscaras:',round(report['seg_map50'],4))
print('Mask R-CNN mAP50-95 máscaras:',round(report['seg_map50_95'],4))
print('Instancias anotadas en test:',report['gt'])
from google.colab import files as colab_files
export=EXP/'CERDOS_07_MaskRCNN_Evaluacion.zip'
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in [CACHED,EXCEL]:z.write(f,arcname=f.name)
    for f in sorted(OUT.glob('*.png')):z.write(f,arcname='ejemplos/'+f.name)
print('✅ ZIP guardado:',export)
colab_files.download(str(export))


**Siguiente:** ejecutar el notebook **08 Comparación (CPU)**. No usar test para nuevos ajustes de modelos.
